# 14 — V3.1 Soil Route Counter Lab

The Frontier Transplant Lab selected **pure Soil**:

- robust score: `0.6590`
- mean family win rate: `0.8056`
- worst family: **Adaptive/3094**, `0.375`
- passive cash: `$171,985`
- zero invalid games

Every broad transplant was worse, so V3.1 stops splicing whole policies.

This notebook preserves Soil's exact farmer/hand route and searches only tiny market-side residuals:
premium-product flood detection, safe sell deferral, price guards, and sell-slot placement.

### Anti-overfit split
Stage 1 targets **Adaptive**, while **3094 is hidden**.
Stage 2 introduces 3094 as an unseen sibling of that lineage and restores the full family-balanced meta.

Promote only if the candidate:
1. improves Adaptive/3094 by at least `+0.10`;
2. keeps global robust score within `-0.01` of pure Soil;
3. retains at least 97% of Soil passive cash;
4. has zero invalid games.

Otherwise the notebook outputs pure Soil again.

## Inputs
Attach: Soil, Adaptive, 3094, V16, Ranker, Melon, Strict Future, Findings.

## Settings
- Accelerator: None / CPU
- Internet: ON


In [ ]:
from pathlib import Path
import subprocess,sys,os,json

WORK=Path('/kaggle/working/v31_soil_counter')
REPO=Path('/kaggle/working/kaggriculture-v31-source')
if not REPO.exists():
    subprocess.run(
        ['git','clone','--depth','1','https://github.com/sidhulyalkar/kaggriculture.git',str(REPO)],
        check=True
    )
commit=subprocess.run(['git','-C',str(REPO),'rev-parse','HEAD'],
                      capture_output=True,text=True,check=True).stdout.strip()
print('repo commit:',commit)
print('cpu:',os.cpu_count())
script=REPO/'scripts'/'soil_route_counter_lab.py'
if not script.exists():
    raise FileNotFoundError(script)


In [ ]:
cmd=[
    sys.executable,str(script),
    '--input-root','/kaggle/input',
    '--work',str(WORK),
    '--workers',str(min(4,os.cpu_count() or 2)),
]
subprocess.run(cmd,check=True)


In [ ]:
import pandas as pd,json
scores=pd.read_csv(WORK/'v31_heldout_scores.csv')
display(scores)
fam=pd.read_csv(WORK/'v31_family_matrix.csv')
display(fam.pivot(index='candidate',columns='family',values='score'))

m=json.loads((WORK/'NEXT_SUBMIT_v31_manifest.json').read_text())
print(json.dumps({
    'selected_candidate':m['selected_candidate'],
    'selection_reason':m['selection_reason'],
    'runner_up':m['runner_up'],
    'next_submission':m['next_submission'],
},indent=2))


## Submit
`/kaggle/working/v31_soil_counter/NEXT_SUBMIT_v31.tar.gz`

Return:
- `NEXT_SUBMIT_v31_manifest.json`
- `v31_heldout_scores.csv`
- `v31_family_matrix.csv`
